# NSEMI Capstone — Script 3: Data Cleaning

**National Semiconductor Ecosystem Maturity Index (NSEMI)**  
Avinash Kashi Venugopal | Walsh College | QM640 | April 2026

This notebook performs **source-level data cleaning** on the 32 raw CSVs extracted in Script 1.  
Cleaning is applied **per source** to preserve traceability. Merging into RQ-specific panels happens in Script 4.

| Operation | Applied to |
|-----------|------------|
| Column name standardization (lowercase, snake_case) | All 32 CSVs |
| State name standardization | RQ2/RQ3 India sources |
| ISO3 country code standardization | Cross-country sources |
| Date parsing (year_month, year) | RQ1, RQ3, RQ4 sources |
| Row filtering (Division 26 for IIP, NIC-26 for PLFS) | MOSPI, PLFS |
| Null-heavy column drop (>80% null) | High-null sources (MOSPI, CEA) |
| Stationarity tests (ADF, KPSS) | RQ1 time series — FLAG only, don't transform |
| KMO/Bartlett readiness | RQ4 cost dimensions — FLAG only |
| Cleaning report generation | All 32 CSVs |

> ⚠️ **Run after Script 1 has completed and 32 CSVs are in Google Drive.**  
> Output: cleaned files saved to `data/cleaned/` folder on Drive.


## 0. Setup


In [1]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Install dependencies (statsmodels for stationarity tests)
!pip install -q statsmodels factor_analyzer

import os, sys, json, warnings
from datetime import datetime
from pathlib import Path
import pandas as pd
import numpy as np
warnings.filterwarnings('ignore')

print(f"Pandas: {pd.__version__}")
print(f"NumPy: {np.__version__}")


Mounted at /content/drive
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.8/42.8 kB 2.5 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
Pandas: 2.2.2
NumPy: 2.0.2


## 0.1 Directory Setup


In [2]:
# Source paths (where Script 1 saved data)
DRIVE_BASE = Path("/content/drive/MyDrive/Walsh_Masters/Term-2_Capstone")
DRIVE_RAW = DRIVE_BASE / "raw"
DRIVE_PROC = DRIVE_BASE / "processed"

# Output path (new cleaned folder)
DRIVE_CLEAN = DRIVE_BASE / "cleaned"
DRIVE_CLEAN.mkdir(parents=True, exist_ok=True)

# Verify source data exists
raw_csvs = sorted(DRIVE_RAW.glob("*.csv")) if DRIVE_RAW.exists() else []
proc_csvs = sorted(DRIVE_PROC.glob("*.csv")) if DRIVE_PROC.exists() else []
print(f"Source CSVs found:")
print(f"  raw/      : {len(raw_csvs)}")
print(f"  processed/: {len(proc_csvs)}")
print(f"  TOTAL     : {len(raw_csvs) + len(proc_csvs)}")
print()
print(f"Output: {DRIVE_CLEAN}")


Source CSVs found:
  raw/      : 31
  processed/: 1
  TOTAL     : 32

Output: /content/drive/MyDrive/Walsh_Masters/Term-2_Capstone/cleaned


## 0.2 Cleaning Utilities & Standardization Tables


In [3]:
# State name standardization (handles common variants)
INDIA_STATES_CANONICAL = {
    'andhra pradesh': 'Andhra Pradesh', 'ap': 'Andhra Pradesh',
    'arunachal pradesh': 'Arunachal Pradesh',
    'assam': 'Assam',
    'bihar': 'Bihar',
    'chhattisgarh': 'Chhattisgarh', 'chattisgarh': 'Chhattisgarh',
    'goa': 'Goa',
    'gujarat': 'Gujarat',
    'haryana': 'Haryana',
    'himachal pradesh': 'Himachal Pradesh', 'hp': 'Himachal Pradesh',
    'jharkhand': 'Jharkhand',
    'karnataka': 'Karnataka',
    'kerala': 'Kerala',
    'madhya pradesh': 'Madhya Pradesh', 'mp': 'Madhya Pradesh',
    'maharashtra': 'Maharashtra',
    'manipur': 'Manipur',
    'meghalaya': 'Meghalaya',
    'mizoram': 'Mizoram',
    'nagaland': 'Nagaland',
    'odisha': 'Odisha', 'orissa': 'Odisha',
    'punjab': 'Punjab',
    'rajasthan': 'Rajasthan',
    'sikkim': 'Sikkim',
    'tamil nadu': 'Tamil Nadu', 'tamilnadu': 'Tamil Nadu', 'tn': 'Tamil Nadu',
    'telangana': 'Telangana', 'ts': 'Telangana',
    'tripura': 'Tripura',
    'uttar pradesh': 'Uttar Pradesh', 'up': 'Uttar Pradesh',
    'uttarakhand': 'Uttarakhand', 'uttaranchal': 'Uttarakhand',
    'west bengal': 'West Bengal', 'wb': 'West Bengal',
}

def standardize_state(name):
    if pd.isna(name):
        return name
    key = str(name).lower().strip()
    return INDIA_STATES_CANONICAL.get(key, name)


# ISO3 country code standardization
COUNTRY_TO_ISO3 = {
    'india': 'IND', 'ind': 'IND',
    'china': 'CHN', 'chn': 'CHN', 'china, mainland': 'CHN',
    'taiwan': 'TWN', 'twn': 'TWN', 'taiwan, china': 'TWN', 'taiwan, province of china': 'TWN', "chinese taipei": 'TWN',
    'south korea': 'KOR', 'korea': 'KOR', 'kor': 'KOR', 'korea, rep.': 'KOR', 'korea, republic of': 'KOR',
    'malaysia': 'MYS', 'mys': 'MYS',
    'vietnam': 'VNM', 'viet nam': 'VNM', 'vnm': 'VNM',
    'japan': 'JPN', 'jpn': 'JPN',
    'germany': 'DEU', 'deu': 'DEU',
    'united states': 'USA', 'usa': 'USA', 'us': 'USA', 'united states of america': 'USA',
    'european union': 'EUR', 'eu': 'EUR',
    'singapore': 'SGP', 'sgp': 'SGP',
}

def standardize_iso3(name):
    if pd.isna(name):
        return name
    key = str(name).lower().strip()
    return COUNTRY_TO_ISO3.get(key, name if len(str(name)) == 3 else None)


# Column name standardization
def clean_column_name(col):
    """Convert to lowercase snake_case"""
    import re
    s = str(col).strip().lower()
    s = re.sub(r'[^a-z0-9_]+', '_', s)
    s = re.sub(r'_+', '_', s)
    s = s.strip('_')
    return s

def standardize_columns(df):
    df = df.copy()
    df.columns = [clean_column_name(c) for c in df.columns]
    return df


# Drop columns with >threshold null rate
def drop_null_heavy_columns(df, threshold=0.80, protect=None):
    df = df.copy()
    protect = protect or []
    null_rate = df.isnull().mean()
    cols_to_drop = [c for c in null_rate.index
                    if null_rate[c] > threshold and c not in protect]
    df = df.drop(columns=cols_to_drop)
    return df, cols_to_drop


print("✓ Cleaning utilities loaded:")
print(f"  - {len(INDIA_STATES_CANONICAL)} state name mappings")
print(f"  - {len(COUNTRY_TO_ISO3)} country to ISO3 mappings")
print("  - Column name standardizer")
print("  - Null-heavy column dropper")


✓ Cleaning utilities loaded:
  - 39 state name mappings
  - 32 country to ISO3 mappings
  - Column name standardizer
  - Null-heavy column dropper


## 0.3 Cleaning Report Tracker


In [4]:
# Track every cleaning operation for the report
cleaning_report = []

def record_cleaning(filename, rq, original_rows, original_cols,
                    cleaned_rows, cleaned_cols, operations_applied,
                    columns_dropped=None, rows_filtered_reason=None,
                    flags=None):
    """Append a cleaning record"""
    cleaning_report.append({
        'filename': filename,
        'rq': rq,
        'original_rows': original_rows,
        'original_cols': original_cols,
        'cleaned_rows': cleaned_rows,
        'cleaned_cols': cleaned_cols,
        'rows_changed': cleaned_rows - original_rows,
        'cols_changed': cleaned_cols - original_cols,
        'operations_applied': '; '.join(operations_applied),
        'columns_dropped': '; '.join(columns_dropped) if columns_dropped else '',
        'rows_filtered_reason': rows_filtered_reason or '',
        'flags': '; '.join(flags) if flags else '',
        'cleaning_date': datetime.now().strftime('%Y-%m-%d %H:%M:%S')
    })

def save_cleaned(df, filename, rq, original_rows, original_cols,
                 operations_applied, columns_dropped=None,
                 rows_filtered_reason=None, flags=None):
    """Save cleaned dataframe and record the operation"""
    output_path = DRIVE_CLEAN / filename
    df.to_csv(output_path, index=False)

    record_cleaning(filename, rq, original_rows, original_cols,
                    len(df), len(df.columns),
                    operations_applied, columns_dropped,
                    rows_filtered_reason, flags)

    print(f"  ✓ Saved: {filename}")
    print(f"    {original_rows} × {original_cols} → {len(df)} × {len(df.columns)} rows × cols")
    if columns_dropped:
        print(f"    Dropped {len(columns_dropped)} null-heavy columns")
    if flags:
        for f in flags:
            print(f"    ⚠ {f}")

print("✓ Cleaning report tracker initialized")


✓ Cleaning report tracker initialized


---
## RQ1 — Ecosystem Depth (4 sources)

Cleaning targets:
- DGFT TRADESTAT: standardize columns, parse year_month, ISO3 codes
- UN Comtrade: standardize columns, ISO3 codes
- MOSPI IIP: filter to Division 26, drop null-heavy columns
- MeitY Electronics: standardize columns


### RQ1.1 DGFT TRADESTAT


In [5]:
# DGFT TRADESTAT — monthly imports by HS code × country
filename = "rq1_dgft_tradestat.csv"
df = pd.read_csv(DRIVE_RAW / filename, low_memory=False)
orig_rows, orig_cols = len(df), len(df.columns)
ops = []

# Standardize column names
df = standardize_columns(df)
ops.append('column_standardization')

# Standardize country names to ISO3 (if country column exists)
if 'country' in df.columns:
    df['country_iso3'] = df['country'].apply(standardize_iso3)
    ops.append('iso3_standardization')

# Parse year_month
if 'year' in df.columns and 'month' in df.columns:
    df['year_month'] = (df['year'].astype(str) + '-' +
                        df['month'].astype(str).str.zfill(2))
    ops.append('year_month_creation')

# Drop null-heavy columns
df, dropped = drop_null_heavy_columns(df, threshold=0.80,
                                       protect=['country', 'country_iso3', 'year', 'month', 'year_month', 'hs_code'])

save_cleaned(df, filename, 'RQ1', orig_rows, orig_cols, ops, dropped)


  ✓ Saved: rq1_dgft_tradestat.csv
    26 × 8 → 26 × 10 rows × cols


### RQ1.2 UN Comtrade


In [6]:
# UN Comtrade — annual cross-country trade data
filename = "rq1_comtrade_crosscountry.csv"
df = pd.read_csv(DRIVE_RAW / filename, low_memory=False)
orig_rows, orig_cols = len(df), len(df.columns)
ops = []

df = standardize_columns(df)
ops.append('column_standardization')

# Standardize reporter to ISO3
for col in ['reporter_name', 'reporter', 'reporterdesc', 'partner_name']:
    if col in df.columns:
        new_col = col.replace('_name', '_iso3').replace('reporterdesc', 'reporter_iso3')
        df[new_col] = df[col].apply(standardize_iso3)
        ops.append(f'iso3_for_{col}')

df, dropped = drop_null_heavy_columns(df, threshold=0.80,
                                       protect=['year', 'hs_code', 'reporter_iso3', 'partner_iso3'])

save_cleaned(df, filename, 'RQ1', orig_rows, orig_cols, ops, dropped)


  ✓ Saved: rq1_comtrade_crosscountry.csv
    2818 × 10 → 2818 × 11 rows × cols


### RQ1.3 MOSPI IIP Division 26


In [7]:
# MOSPI IIP — filter to Division 26 (Computer, Electronic, Optical Products)
filename = "rq1_mospi_iip_div26.csv"
df = pd.read_csv(DRIVE_RAW / filename, low_memory=False)
orig_rows, orig_cols = len(df), len(df.columns)
ops = []
flags = []

df = standardize_columns(df)
ops.append('column_standardization')

# Filter to Division 26 rows if NIC code column exists
filter_reason = None
if '_nic_2008' in df.columns:
    div26_mask = df['_nic_2008'].astype(str).str.strip() == '26'
    if div26_mask.sum() > 0:
        df = df[div26_mask].copy()
        ops.append('filter_to_division_26')
        filter_reason = f'Filtered to NIC 2008 = 26 (n={div26_mask.sum()})'
elif 'description' in df.columns:
    desc_mask = df['description'].str.contains('Computer|Electronic|Optical', case=False, na=False)
    if desc_mask.sum() > 0:
        df = df[desc_mask].copy()
        ops.append('filter_to_division_26_by_description')
        filter_reason = f'Filtered by description match (n={desc_mask.sum()})'
else:
    flags.append('Division 26 filter could not be applied — no NIC code column found')

# Drop null-heavy columns
df, dropped = drop_null_heavy_columns(df, threshold=0.80,
                                       protect=['_nic_2008', 'description', 'month', '_year'])

# Flag if too few rows after filter
if len(df) < 10:
    flags.append(f'Only {len(df)} rows after Division 26 filter — investigate')

save_cleaned(df, filename, 'RQ1', orig_rows, orig_cols, ops, dropped,
             filter_reason, flags)


  ✓ Saved: rq1_mospi_iip_div26.csv
    6 × 8 → 6 × 8 rows × cols
    ⚠ Division 26 filter could not be applied — no NIC code column found
    ⚠ Only 6 rows after Division 26 filter — investigate


In [9]:
import pandas as pd
from pathlib import Path

DRIVE_RAW = Path("/content/drive/MyDrive/Walsh_Masters/Term-2_Capstone/raw")
DRIVE_CLEAN = Path("/content/drive/MyDrive/Walsh_Masters/Term-2_Capstone/cleaned")

print("=" * 70)
print("DIAGNOSTIC: rq1_mospi_iip_div26.csv")
print("=" * 70)

# Read the raw file (before cleaning)
df_raw = pd.read_csv(DRIVE_RAW / "rq1_mospi_iip_div26.csv", low_memory=False)
print(f"\nRAW file dimensions: {len(df_raw)} rows × {len(df_raw.columns)} cols")
print(f"\nAll columns ({len(df_raw.columns)}):")
for c in df_raw.columns:
    null_pct = df_raw[c].isnull().mean() * 100
    sample = df_raw[c].dropna().head(2).tolist()
    print(f"  {c:50s} | nulls: {null_pct:5.1f}% | sample: {sample}")

print(f"\nFirst 10 rows preview:")
print(df_raw.head(10).to_string())

print(f"\nUnique values in 'dataset' column (if exists):")
if 'dataset' in df_raw.columns:
    print(df_raw['dataset'].value_counts().to_string())

# Check for IIP-specific columns
print(f"\nLooking for IIP-related columns:")
iip_cols = [c for c in df_raw.columns if any(k in c.lower() for k in ['iip', 'index', 'value', 'gen', 'manuf', 'mining'])]
for c in iip_cols:
    print(f"  Found: {c}")

# Check the cleaned version
print(f"\n" + "=" * 70)
print("CLEANED file (after cleaning script):")
df_clean = pd.read_csv(DRIVE_CLEAN / "rq1_mospi_iip_div26.csv", low_memory=False)
print(f"Dimensions: {len(df_clean)} rows × {len(df_clean.columns)} cols")
print(f"\nFirst 10 rows:")
print(df_clean.head(10).to_string())

DIAGNOSTIC: rq1_mospi_iip_div26.csv

RAW file dimensions: 6 rows × 8 cols

All columns (8):
  s__no_                                             | nulls:   0.0% | sample: [104, 290]
  nic_2008_5_digit                                   | nulls:   0.0% | sample: [18200, 26204]
  item_groups                                        | nulls:   0.0% | sample: ['Digital media for electronic media players', 'Computers & peripherals']
  unit                                               | nulls:   0.0% | sample: ['Th.Nos.', 'Rs.Crore']
  use_based_classification                           | nulls:   0.0% | sample: ['Consumer durables', 'Consumer durables']
  weights__in___                                     | nulls:   0.0% | sample: [0.015, 0.1676]
  data_source                                        | nulls:   0.0% | sample: ['MOSPI_IIP_API_real', 'MOSPI_IIP_API_real']
  retrieval_date                                     | nulls:   0.0% | sample: ['2026-04-26', '2026-04-26']

First 10 rows prev

### RQ1.4 MeitY Electronics


In [10]:
import pandas as pd
from pathlib import Path
DRIVE_RAW = Path("/content/drive/MyDrive/Walsh_Masters/Term-2_Capstone/raw")
df = pd.read_csv(DRIVE_RAW / "rq1_meity_electronics.csv", low_memory=False)
print(f"Total columns: {len(df.columns)}")
print(f"Unique columns: {df.columns.nunique()}")
print(f"Duplicates: {df.columns[df.columns.duplicated()].tolist()}")
print(f"\nAll columns:")
for c in df.columns:
    print(f"  {c}")

Total columns: 19
Unique columns: 19
Duplicates: []

All columns:
  _sl__no_
  item_vertical
  __2013_14
  _2014_15_
  __2015_16
  __2016_17
  _2017_18
  _2018_19
  _2019_20
  _2020_21_
  resource_id
  _year
  production_
  import
  export
  __2014_15
  _2020_21
  data_source
  retrieval_date


In [11]:
def drop_null_heavy_columns(df, threshold=0.80, protect=None):
    """Drop columns with >threshold null rate. Handles duplicate column names."""
    df = df.copy()
    protect = protect or []

    # Handle duplicate column names by deduplicating first
    if df.columns.duplicated().any():
        # Keep the first occurrence of each duplicated name
        df = df.loc[:, ~df.columns.duplicated()]

    null_rate = df.isnull().mean()
    cols_to_drop = []
    for c in null_rate.index:
        rate = null_rate[c]
        # Handle case where rate could be a Series (shouldn't happen after dedup, but defensive)
        if hasattr(rate, '__len__'):
            rate = rate.iloc[0] if len(rate) > 0 else 0
        if rate > threshold and c not in protect:
            cols_to_drop.append(c)

    df = df.drop(columns=cols_to_drop)
    return df, cols_to_drop


print("✓ Updated drop_null_heavy_columns to handle duplicate column names")

✓ Updated drop_null_heavy_columns to handle duplicate column names


In [12]:
# MeitY Electronics — annual production
filename = "rq1_meity_electronics.csv"
df = pd.read_csv(DRIVE_RAW / filename, low_memory=False)
orig_rows, orig_cols = len(df), len(df.columns)
ops = []

df = standardize_columns(df)
ops.append('column_standardization')

df, dropped = drop_null_heavy_columns(df, threshold=0.80,
                                       protect=['year', 'item_vertical', 'production_value'])

save_cleaned(df, filename, 'RQ1', orig_rows, orig_cols, ops, dropped)


  ✓ Saved: rq1_meity_electronics.csv
    14 × 19 → 14 × 17 rows × cols


---
## RQ2 — Infrastructure Reliability (5 sources + 1 shared)

Cleaning targets:
- ISM Projects: standardize state names
- CEA Power: standardize columns, drop null-heavy
- SERC Tariffs: standardize state names
- Power Ministry: standardize columns
- IMD Weather: standardize columns
- Facility Locations: standardize state names


### RQ2.1 ISM Projects (Shared with RQ4)


In [13]:
# ISM Projects — used for binary ism_approved flag in RQ2
filename = "ism_projects.csv"
df = pd.read_csv(DRIVE_RAW / filename, low_memory=False)
orig_rows, orig_cols = len(df), len(df.columns)
ops = []

df = standardize_columns(df)
ops.append('column_standardization')

if 'state' in df.columns:
    df['state'] = df['state'].apply(standardize_state)
    ops.append('state_name_standardization')

save_cleaned(df, filename, 'RQ2/Shared', orig_rows, orig_cols, ops)


  ✓ Saved: ism_projects.csv
    7 × 13 → 7 × 13 rows × cols


### RQ2.2 CEA Power Statistics


In [14]:
# CEA Power — state-level deficit, T&D, capacity
filename = "rq2_cea_power_raw.csv"
df = pd.read_csv(DRIVE_RAW / filename, low_memory=False)
orig_rows, orig_cols = len(df), len(df.columns)
ops = []
flags = []

df = standardize_columns(df)
ops.append('column_standardization')

# Find state column
state_col = None
for c in ['state_u_ts_', 'state', 'region_state_u_t_', 'state_system_region',
         'state_ut', 'states_uts']:
    if c in df.columns and df[c].notna().sum() > 5:
        state_col = c
        break

if state_col:
    df[state_col] = df[state_col].apply(standardize_state)
    if state_col != 'state':
        df = df.rename(columns={state_col: 'state'})
        ops.append(f'rename_{state_col}_to_state')
    ops.append('state_name_standardization')
else:
    flags.append('No state column found — investigate column structure')

# CEA has very high null rate (~87%) — drop aggressively
df, dropped = drop_null_heavy_columns(df, threshold=0.80, protect=['state'])

save_cleaned(df, filename, 'RQ2', orig_rows, orig_cols, ops, dropped, flags=flags)


  ✓ Saved: rq2_cea_power_raw.csv
    53 × 21 → 53 × 8 rows × cols
    Dropped 13 null-heavy columns
    ⚠ No state column found — investigate column structure


In [15]:
import pandas as pd
from pathlib import Path

DRIVE_RAW = Path("/content/drive/MyDrive/Walsh_Masters/Term-2_Capstone/raw")
DRIVE_CLEAN = Path("/content/drive/MyDrive/Walsh_Masters/Term-2_Capstone/cleaned")

print("=" * 70)
print("DIAGNOSTIC: rq2_cea_power_raw.csv")
print("=" * 70)

# Read the raw file
df_raw = pd.read_csv(DRIVE_RAW / "rq2_cea_power_raw.csv", low_memory=False)
print(f"\nRAW file: {len(df_raw)} rows × {len(df_raw.columns)} cols")

print(f"\nAll {len(df_raw.columns)} columns with null %:")
for c in df_raw.columns:
    null_pct = df_raw[c].isnull().mean() * 100
    sample_vals = df_raw[c].dropna().head(3).astype(str).tolist()
    print(f"  {c[:60]:60s} | nulls: {null_pct:5.1f}% | sample: {sample_vals}")

print(f"\nFirst 5 rows:")
print(df_raw.head().to_string())

print(f"\n--- Searching for state-like values across ALL columns ---")
for c in df_raw.columns:
    if df_raw[c].dtype == 'object':
        unique_vals = df_raw[c].dropna().astype(str).str.strip().unique()
        # Check if any look like Indian states
        state_like = [v for v in unique_vals if any(s in v.lower() for s in
                      ['gujarat', 'maharashtra', 'karnataka', 'tamil', 'andhra', 'kerala'])]
        if state_like:
            print(f"  Column '{c}' has state-like values: {state_like[:5]}")
            print(f"    All unique: {unique_vals[:20].tolist()}")

# Check the cleaned version
print(f"\n" + "=" * 70)
print("CLEANED file:")
df_clean = pd.read_csv(DRIVE_CLEAN / "rq2_cea_power_raw.csv", low_memory=False)
print(f"Dimensions: {len(df_clean)} rows × {len(df_clean.columns)} cols")
print(f"\nColumns kept: {list(df_clean.columns)}")
print(f"\nFirst 5 rows:")
print(df_clean.head().to_string())

DIAGNOSTIC: rq2_cea_power_raw.csv

RAW file: 53 rows × 21 cols

All 21 columns with null %:
  state_u_ts_                                                  | nulls:   7.5% | sample: ['Haryana', 'Himachal Pradesh', 'Jammu & Kashmir']
  thermal                                                      | nulls:   7.5% | sample: ['7087.82', '213.9', '633.46']
  nuclear                                                      | nulls:   7.5% | sample: ['109.16', '34.08', '77.0']
  hydro                                                        | nulls:   7.5% | sample: ['1456.83', '3421.51', '2255.21']
  r_e_s                                                        | nulls:   7.5% | sample: ['138.6', '754.8', '156.53']
  _total                                                       | nulls:   7.5% | sample: ['8792.41', '4424.29', '3122.2']
  _year                                                        | nulls:  92.5% | sample: ['2020-21', '2021-22', '2022-23']
  electricity_production_from_fossil_fuel___c

In [42]:
!pip install -q pdfplumber

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 1.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.2/68.2 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 77.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.7/3.7 MB 62.5 MB/s eta 0:00:00


In [44]:
import pdfplumber
from pathlib import Path

# Upload the PDF directly to Drive first (any folder)
PDF_PATH = "/content/drive/MyDrive/Walsh_Masters/Term-2_Capstone/raw/pdfs/Report on Performance of Power Utilities 2024-25.pdf"

with pdfplumber.open(PDF_PATH) as pdf:
    print(f"Total pages: {len(pdf.pages)}")
    print(f"\nFirst page preview:\n")
    print(pdf.pages[0].extract_text()[:2000])

    # Search for table with state-wise T&D losses
    for i, page in enumerate(pdf.pages):
        text = page.extract_text() or ""
        if "T&D" in text or "transmission" in text.lower() or "AT&C" in text:
            print(f"\n--- Page {i+1} contains T&D-related content ---")
            print(text[:3000])
            break

Total pages: 90

First page preview:

REPORT ON
PERFORMANCE OF POWER UTILITIES
2024-25
A MAHARATNA COMPANY
FEBRUARY 2026

--- Page 3 contains T&D-related content ---
Contents
Executive Summary i - iii
Annexures
Section 1: Performance of Distribution Utilities
Annexure 1.1 : Revenue Details 1 – 6
Annexure 1.1(a) : Revenue Structure 7 – 12
Annexure 1.1(b) : Revenue Structure on Energy Sold basis 13 – 18
Annexure 1.2 : Expense Details 19 – 24
Annexure 1.2(a) : Cost Structure 25 – 30
Annexure 1.2(b) : Cost Structure on Energy Sold basis 31 – 36
Annexure 1.3 : Profitability and Gap 37 – 42
Annexure 1.3(a) : Gap on Energy Sold basis 43 – 44
Annexure 1.3(b) : Cash Adjusted Gap 45 – 50
Annexure 1.4 : Total Assets 51 – 56
Annexure 1.5 : Total Equity and Liabilities 57 – 62
Annexure 1.6 : Net Worth, Consumer Contribution & Total Borrowings 63 – 68
Annexure 1.7 : Debt Service Coverage ratio (Cash Adjusted) 69 – 74
Annexure 1.8 : AT&C Losses 75 – 80
Appendix-1 : Status of accounts


In [46]:
import pdfplumber
from pathlib import Path
import pandas as pd

PDF_PATH = "/content/drive/MyDrive/Walsh_Masters/Term-2_Capstone/raw/pdfs/Report on Performance of Power Utilities 2024-25.pdf"

with pdfplumber.open(PDF_PATH) as pdf:
    # Annexure 1.8 is on pages 75-80 per the TOC
    # PDF page numbering may differ from printed page numbering
    # Try pages around 75-80 first; we may need to adjust

    for page_num in range(75, 86):  # Try a wider range to be safe
        if page_num >= len(pdf.pages):
            break
        page = pdf.pages[page_num - 1]  # PDF pages are 0-indexed in pdfplumber
        text = page.extract_text() or ""

        # Print first few lines to identify the right pages
        first_lines = '\n'.join(text.split('\n')[:5])
        print(f"\n--- PDF Page {page_num} ---")
        print(first_lines)

        # If this looks like the AT&C losses page, show full content
        if 'AT&C' in text or 'AT & C' in text:
            print(f"\n*** Page {page_num} appears to contain AT&C losses ***")
            print(text[:3000])
            print("\n--- TABLES on this page ---")
            tables = page.extract_tables()
            print(f"Number of tables found: {len(tables)}")
            for i, table in enumerate(tables):
                print(f"\nTable {i+1} ({len(table)} rows × {len(table[0]) if table else 0} cols):")
                for row in table[:5]:  # Show first 5 rows
                    print(f"  {row}")


--- PDF Page 75 ---
Section 1: Performance of Distribution Utilities Annexure 1.6
Net Worth, Consumer Contribution & Total Borrowings
Rs crore
As on March 31, 2023
Total

--- PDF Page 76 ---
Section 1: Performance of Distribution Utilities Annexure 1.7
Debt Service Coverage Ratio (Cash Adjusted)
As on March 31, 2025
Opening-Current Maturities on
Long Term Borrowings and

--- PDF Page 77 ---
Section 1: Performance of Distribution Utilities Annexure 1.7
Debt Service Coverage Ratio (Cash Adjusted)
As on March 31, 2025
Opening-Current Maturities on
Long Term Borrowings and

--- PDF Page 78 ---
Section 1: Performance of Distribution Utilities Annexure 1.7
Debt Service Coverage Ratio (Cash Adjusted)
As on March 31, 2024
Opening-Current Maturities on
Long Term Borrowings and

--- PDF Page 79 ---
Section 1: Performance of Distribution Utilities Annexure 1.7
Debt Service Coverage Ratio (Cash Adjusted)
As on March 31, 2024
Opening-Current Maturities on
Long Term Borrowings and

--- PDF Page 80 

In [47]:
import pdfplumber
import pandas as pd
from pathlib import Path

PDF_PATH = "/content/drive/MyDrive/Walsh_Masters/Term-2_Capstone/raw/pdfs/Report on Performance of Power Utilities 2024-25.pdf"
DRIVE_RAW = Path("/content/drive/MyDrive/Walsh_Masters/Term-2_Capstone/raw")

# State name canonical list (per your synopsis)
INDIA_STATES = [
    "Andhra Pradesh", "Arunachal Pradesh", "Assam", "Bihar", "Chhattisgarh",
    "Chattisgarh",  # PFC uses this spelling
    "Delhi", "Goa", "Gujarat", "Haryana", "Himachal Pradesh", "Jharkhand",
    "Karnataka", "Kerala", "Madhya Pradesh", "Maharashtra", "Manipur",
    "Meghalaya", "Mizoram", "Nagaland", "Odisha", "Punjab", "Rajasthan",
    "Sikkim", "Tamil Nadu", "Telangana", "Tripura", "Uttar Pradesh",
    "Uttarakhand", "West Bengal",
    # UTs (we'll filter to states only later)
    "Andaman & Nicobar Islands", "Ladakh", "Puducherry",
    "Dadra & Nagar Haveli and Daman & Diu",
]

# Extract data from State Sector pages (82-83 for 2024-25, 84-85 for 2023-24)
state_data = {2024: {}, 2023: {}}

with pdfplumber.open(PDF_PATH) as pdf:
    # Pages are 0-indexed in pdfplumber
    page_year_map = {
        81: 2024,  # Page 82 (printed) = 2024-25 part 1
        82: 2024,  # Page 83 (printed) = 2024-25 part 2
        83: 2023,  # Page 84 (printed) = 2023-24 part 1
        84: 2023,  # Page 85 (printed) = 2023-24 part 2
    }

    for page_idx, year in page_year_map.items():
        page = pdf.pages[page_idx]
        text = page.extract_text() or ""
        lines = text.split('\n')

        for line in lines:
            line = line.strip()
            # Look for lines that start with a state name and have 5 numeric fields after
            for state in INDIA_STATES:
                if line.startswith(state + " "):
                    # The line format is: "STATE NAME N1 N2 N3 N4 N5"
                    # where N5 is AT&C Loss %
                    rest = line[len(state):].strip()
                    parts = rest.split()
                    if len(parts) >= 5:
                        try:
                            # Last value should be AT&C Loss %
                            atc_loss = float(parts[-1].replace(',', ''))
                            # First value (Net Input Energy) sanity check
                            net_input = float(parts[0].replace(',', ''))
                            # Only accept if reasonable (AT&C between 0-70%)
                            if 0 <= atc_loss <= 70:
                                # Use canonical name (Chattisgarh -> Chhattisgarh)
                                canonical = "Chhattisgarh" if state == "Chattisgarh" else state
                                # Don't overwrite (state-level should appear before sub-discoms)
                                if canonical not in state_data[year]:
                                    state_data[year][canonical] = {
                                        'atc_loss_pct': atc_loss,
                                        'net_input_energy_mu': net_input,
                                        'net_energy_sold_mu': float(parts[1].replace(',', '')),
                                        'billing_efficiency_pct': float(parts[2].replace(',', '')),
                                        'collection_efficiency_pct': float(parts[3].replace(',', '')),
                                    }
                        except (ValueError, IndexError):
                            pass
                    break  # Move to next line once state matched

# Build dataframes for each year
for year in [2024, 2023]:
    print(f"\n=== Year {year}-{str(year+1)[2:]} extracted: {len(state_data[year])} states ===")
    for state, vals in sorted(state_data[year].items()):
        print(f"  {state:30s}: AT&C = {vals['atc_loss_pct']:.2f}%")

# Combine into a single CSV with both years (latest as primary)
rows = []
for state, vals_2024 in state_data[2024].items():
    rows.append({
        'state': state,
        'atc_loss_pct_2024_25': vals_2024['atc_loss_pct'],
        'atc_loss_pct_2023_24': state_data[2023].get(state, {}).get('atc_loss_pct'),
        'net_input_energy_mu_2024_25': vals_2024['net_input_energy_mu'],
        'net_energy_sold_mu_2024_25': vals_2024['net_energy_sold_mu'],
        'billing_efficiency_pct_2024_25': vals_2024['billing_efficiency_pct'],
        'collection_efficiency_pct_2024_25': vals_2024['collection_efficiency_pct'],
        'data_source': 'PFC_Performance_Report_2024_25',
        'retrieval_date': pd.Timestamp.now().strftime('%Y-%m-%d'),
    })

df = pd.DataFrame(rows).sort_values('state').reset_index(drop=True)

# Save to raw folder
output_path = DRIVE_RAW / "rq2_pfc_atc_losses.csv"
df.to_csv(output_path, index=False)

print(f"\n{'='*60}")
print(f"✓ Saved: {output_path.name}")
print(f"  {len(df)} states/UTs × {len(df.columns)} columns")
print(f"\nFinal data preview:")
print(df[['state', 'atc_loss_pct_2024_25', 'atc_loss_pct_2023_24']].to_string(index=False))


=== Year 2024-25 extracted: 32 states ===
  Andaman & Nicobar Islands     : AT&C = 24.14%
  Andhra Pradesh                : AT&C = 7.87%
  Arunachal Pradesh             : AT&C = 46.20%
  Assam                         : AT&C = 15.44%
  Bihar                         : AT&C = 15.51%
  Chhattisgarh                  : AT&C = 14.25%
  Delhi                         : AT&C = 8.36%
  Goa                           : AT&C = 10.39%
  Gujarat                       : AT&C = 8.25%
  Haryana                       : AT&C = 11.76%
  Himachal Pradesh              : AT&C = 19.44%
  Jharkhand                     : AT&C = 28.19%
  Karnataka                     : AT&C = 11.92%
  Kerala                        : AT&C = 6.61%
  Ladakh                        : AT&C = 26.82%
  Madhya Pradesh                : AT&C = 22.76%
  Maharashtra                   : AT&C = 17.69%
  Manipur                       : AT&C = 12.90%
  Meghalaya                     : AT&C = 17.52%
  Mizoram                       : AT&C = 32.31%
 

In [48]:
import pandas as pd
from pathlib import Path

DRIVE_RAW = Path("/content/drive/MyDrive/Walsh_Masters/Term-2_Capstone/raw")
DRIVE_CLEAN = Path("/content/drive/MyDrive/Walsh_Masters/Term-2_Capstone/cleaned")

# Read the freshly extracted PFC data
filename = "rq2_pfc_atc_losses.csv"
df = pd.read_csv(DRIVE_RAW / filename, low_memory=False)
orig_rows, orig_cols = len(df), len(df.columns)
ops = []

# Standardize column names (already lowercase but ensure consistency)
df = standardize_columns(df)
ops.append('column_standardization')

# Standardize state names
df['state'] = df['state'].apply(standardize_state)
ops.append('state_name_standardization')

# Add ism_approved flag for RQ2 grouping (the 3 ISM-approved states)
ISM_APPROVED_STATES = ['Gujarat', 'Assam', 'Uttar Pradesh']
df['ism_approved'] = df['state'].isin(ISM_APPROVED_STATES).astype(int)
ops.append('ism_approved_flag_added')

# Save cleaned version
save_cleaned(df, filename, 'RQ2', orig_rows, orig_cols, ops)

# Verify
print(f"\n{'='*60}")
print(f"RQ2 ISM vs Non-ISM AT&C Loss preview:")
print(f"{'='*60}")
ism_subset = df[df['ism_approved'] == 1][['state', 'atc_loss_pct_2024_25', 'atc_loss_pct_2023_24']]
non_ism_subset = df[df['ism_approved'] == 0][['state', 'atc_loss_pct_2024_25']]

print(f"\nISM-Approved States (n={len(ism_subset)}):")
print(ism_subset.to_string(index=False))
print(f"  Mean AT&C 2024-25: {ism_subset['atc_loss_pct_2024_25'].mean():.2f}%")
print(f"  Median AT&C 2024-25: {ism_subset['atc_loss_pct_2024_25'].median():.2f}%")

print(f"\nNon-ISM States (n={len(non_ism_subset)}):")
print(f"  Mean AT&C 2024-25: {non_ism_subset['atc_loss_pct_2024_25'].mean():.2f}%")
print(f"  Median AT&C 2024-25: {non_ism_subset['atc_loss_pct_2024_25'].median():.2f}%")

print(f"\n{'='*60}")
print("This is preliminary descriptive comparison.")
print("Mann-Whitney U test will be run in Script 5 (Modeling).")
print(f"{'='*60}")

  ✓ Saved: rq2_pfc_atc_losses.csv
    32 × 9 → 32 × 10 rows × cols

RQ2 ISM vs Non-ISM AT&C Loss preview:

ISM-Approved States (n=3):
        state  atc_loss_pct_2024_25  atc_loss_pct_2023_24
        Assam                 15.44                 14.03
      Gujarat                  8.25                  8.97
Uttar Pradesh                 19.54                 16.39
  Mean AT&C 2024-25: 14.41%
  Median AT&C 2024-25: 15.44%

Non-ISM States (n=29):
  Mean AT&C 2024-25: 19.48%
  Median AT&C 2024-25: 17.52%

This is preliminary descriptive comparison.
Mann-Whitney U test will be run in Script 5 (Modeling).


### RQ2.3 SERC Tariffs


In [16]:
# SERC Tariffs — state-level industrial electricity rates
filename = "rq2_serc_tariffs.csv"
df = pd.read_csv(DRIVE_RAW / filename, low_memory=False)
orig_rows, orig_cols = len(df), len(df.columns)
ops = []

df = standardize_columns(df)
ops.append('column_standardization')

state_col = None
for c in ['state_ut', 'states_ut', 'states__discoms', 'state']:
    if c in df.columns and df[c].notna().sum() > 5:
        state_col = c
        break

if state_col:
    df[state_col] = df[state_col].apply(standardize_state)
    if state_col != 'state':
        df = df.rename(columns={state_col: 'state'})
    ops.append('state_name_standardization')

df, dropped = drop_null_heavy_columns(df, threshold=0.80, protect=['state'])

save_cleaned(df, filename, 'RQ2', orig_rows, orig_cols, ops, dropped)


  ✓ Saved: rq2_serc_tariffs.csv
    42 × 10 → 42 × 6 rows × cols
    Dropped 4 null-heavy columns


### RQ2.4 Power Ministry


In [17]:
# Power Ministry sector at a glance (national-level)
filename = "rq2_powermin_sector_glance.csv"
df = pd.read_csv(DRIVE_RAW / filename, low_memory=False)
orig_rows, orig_cols = len(df), len(df.columns)
ops = []

df = standardize_columns(df)
ops.append('column_standardization')

df, dropped = drop_null_heavy_columns(df, threshold=0.80)

save_cleaned(df, filename, 'RQ2', orig_rows, orig_cols, ops, dropped)


  ✓ Saved: rq2_powermin_sector_glance.csv
    53 × 22 → 53 × 9 rows × cols
    Dropped 13 null-heavy columns


### RQ2.5 IMD Weather


In [18]:
# IMD Weather — facility location climate
filename = "rq2_imd_weather.csv"
df = pd.read_csv(DRIVE_RAW / filename, low_memory=False)
orig_rows, orig_cols = len(df), len(df.columns)
ops = []

df = standardize_columns(df)
ops.append('column_standardization')

# Parse date
if 'date' in df.columns or 'time' in df.columns:
    date_col = 'date' if 'date' in df.columns else 'time'
    df[date_col] = pd.to_datetime(df[date_col], errors='coerce')
    ops.append('date_parsing')

df, dropped = drop_null_heavy_columns(df, threshold=0.80,
                                       protect=['date', 'time', 'location'])

save_cleaned(df, filename, 'RQ2', orig_rows, orig_cols, ops, dropped)


  ✓ Saved: rq2_imd_weather.csv
    1 × 4 → 1 × 4 rows × cols


### RQ2.6 Semiconductor Facility Locations


In [19]:
# Facility locations — derived reference table from ISM
filename = "rq2_semiconductor_facility_locations.csv"
df = pd.read_csv(DRIVE_RAW / filename, low_memory=False)
orig_rows, orig_cols = len(df), len(df.columns)
ops = []

df = standardize_columns(df)
ops.append('column_standardization')

if 'state' in df.columns:
    df['state'] = df['state'].apply(standardize_state)
    ops.append('state_name_standardization')

save_cleaned(df, filename, 'RQ2', orig_rows, orig_cols, ops)


  ✓ Saved: rq2_semiconductor_facility_locations.csv
    7 × 6 → 7 × 6 rows × cols


---
## RQ3 — Workforce Readiness (9 sources)

Cleaning targets:
- UNESCO UIS: standardize ISO3, year
- World Bank Talent: pivot from long if needed, standardize ISO3
- World Bank Workforce: pivot from long if needed, standardize ISO3
- AICTE Intake: standardize state names
- AICTE Skilled Talent: standardize columns
- PLFS Employment: filter to NIC-26
- NASSCOM GCC: standardize columns
- Electronics Production Exports: standardize columns
- IESA Demand Projections: compiled reference (light cleaning)
- Workforce Compiled: compiled reference (light cleaning)


### RQ3.1 UNESCO UIS


In [20]:
# UNESCO UIS — cross-country tertiary enrollment
filename = "rq3_unesco_graduates.csv"
df = pd.read_csv(DRIVE_RAW / filename, low_memory=False)
orig_rows, orig_cols = len(df), len(df.columns)
ops = []

df = standardize_columns(df)
ops.append('column_standardization')

# Standardize ISO3 if needed
if 'country' in df.columns and 'country_iso3' not in df.columns:
    df['country_iso3'] = df['country'].apply(standardize_iso3)
    ops.append('iso3_standardization')

df, dropped = drop_null_heavy_columns(df, threshold=0.80,
                                       protect=['year', 'country_iso3'])

save_cleaned(df, filename, 'RQ3', orig_rows, orig_cols, ops, dropped)


  ✓ Saved: rq3_unesco_graduates.csv
    48 × 7 → 48 × 7 rows × cols


### RQ3.2 World Bank Talent Indicators


In [21]:
# World Bank Talent (R&D, patents, tertiary enrollment)
filename = "rq3_worldbank_talent_indicators.csv"
df = pd.read_csv(DRIVE_RAW / filename, low_memory=False)
orig_rows, orig_cols = len(df), len(df.columns)
ops = []

df = standardize_columns(df)
ops.append('column_standardization')

# Already in long format with iso3 column from World Bank API
if 'iso3' in df.columns and 'country_iso3' not in df.columns:
    df = df.rename(columns={'iso3': 'country_iso3'})
    ops.append('rename_iso3_to_country_iso3')

df, dropped = drop_null_heavy_columns(df, threshold=0.80,
                                       protect=['country_iso3', 'year', 'indicator', 'value'])

save_cleaned(df, filename, 'RQ3', orig_rows, orig_cols, ops, dropped)


  ✓ Saved: rq3_worldbank_talent_indicators.csv
    271 × 7 → 271 × 7 rows × cols


### RQ3.3 World Bank Workforce Indicators


In [22]:
# World Bank Workforce
filename = "rq3_worldbank_workforce_indicators.csv"
df = pd.read_csv(DRIVE_RAW / filename, low_memory=False)
orig_rows, orig_cols = len(df), len(df.columns)
ops = []

df = standardize_columns(df)
ops.append('column_standardization')

if 'iso3' in df.columns and 'country_iso3' not in df.columns:
    df = df.rename(columns={'iso3': 'country_iso3'})
    ops.append('rename_iso3_to_country_iso3')

df, dropped = drop_null_heavy_columns(df, threshold=0.80,
                                       protect=['country_iso3', 'year', 'indicator', 'value'])

save_cleaned(df, filename, 'RQ3', orig_rows, orig_cols, ops, dropped)


  ✓ Saved: rq3_worldbank_workforce_indicators.csv
    210 × 7 → 210 × 7 rows × cols


### RQ3.4 AICTE Intake


In [23]:
# AICTE Engineering Intake — state-level
filename = "rq3_aicte_intake.csv"
df = pd.read_csv(DRIVE_RAW / filename, low_memory=False)
orig_rows, orig_cols = len(df), len(df.columns)
ops = []

df = standardize_columns(df)
ops.append('column_standardization')

# Find state column
state_col = None
for c in ['state__ut_name', 'state', 'state_name', 'state_ut']:
    if c in df.columns and df[c].notna().sum() > 5:
        state_col = c
        break

if state_col:
    df[state_col] = df[state_col].apply(standardize_state)
    if state_col != 'state':
        df = df.rename(columns={state_col: 'state'})
    ops.append('state_name_standardization')

df, dropped = drop_null_heavy_columns(df, threshold=0.80, protect=['state'])

save_cleaned(df, filename, 'RQ3', orig_rows, orig_cols, ops, dropped)


  ✓ Saved: rq3_aicte_intake.csv
    6 × 11 → 6 × 11 rows × cols


### RQ3.5 AICTE Electronics Skilled Talent


In [24]:
# AICTE Electronics Skilled Talent
filename = "rq3_aicte_electronics_skilled_talent.csv"
df = pd.read_csv(DRIVE_RAW / filename, low_memory=False)
orig_rows, orig_cols = len(df), len(df.columns)
ops = []

df = standardize_columns(df)
ops.append('column_standardization')

# State standardization if applicable
for c in ['state', 'state_ut', 'state_name']:
    if c in df.columns:
        df[c] = df[c].apply(standardize_state)
        ops.append('state_name_standardization')
        break

df, dropped = drop_null_heavy_columns(df, threshold=0.80)

save_cleaned(df, filename, 'RQ3', orig_rows, orig_cols, ops, dropped)


  ✓ Saved: rq3_aicte_electronics_skilled_talent.csv
    39 × 22 → 39 × 18 rows × cols
    Dropped 4 null-heavy columns


### RQ3.6 PLFS Employment


In [25]:
# PLFS Employment — filter to NIC-26 (electronics manufacturing)
filename = "rq3_plfs_employment.csv"
df = pd.read_csv(DRIVE_RAW / filename, low_memory=False)
orig_rows, orig_cols = len(df), len(df.columns)
ops = []
flags = []

df = standardize_columns(df)
ops.append('column_standardization')

# Filter to NIC-26 rows if column exists
filter_reason = None
if 'nic_26' in df.columns:
    nic26_mask = df['nic_26'].notna()
    if nic26_mask.sum() > 0:
        df = df[nic26_mask].copy()
        ops.append('filter_to_NIC_26')
        filter_reason = f'Filtered to rows with NIC 26 data (n={nic26_mask.sum()})'
elif 'nic' in df.columns:
    nic_mask = df['nic'].astype(str).str.startswith('26')
    if nic_mask.sum() > 0:
        df = df[nic_mask].copy()
        ops.append('filter_to_NIC_26')
        filter_reason = f'Filtered by NIC code starting 26 (n={nic_mask.sum()})'
else:
    flags.append('No NIC column found — keeping all rows')

df, dropped = drop_null_heavy_columns(df, threshold=0.80, protect=['nic_26', 'nic'])

save_cleaned(df, filename, 'RQ3', orig_rows, orig_cols, ops, dropped,
             filter_reason, flags)


  ✓ Saved: rq3_plfs_employment.csv
    78 × 45 → 31 × 34 rows × cols
    Dropped 10 null-heavy columns


### RQ3.7 NASSCOM GCC Supply


In [26]:
# NASSCOM GCC Supply
filename = "rq3_nasscom_gcc_supply.csv"
df = pd.read_csv(DRIVE_RAW / filename, low_memory=False)
orig_rows, orig_cols = len(df), len(df.columns)
ops = []

df = standardize_columns(df)
ops.append('column_standardization')

df, dropped = drop_null_heavy_columns(df, threshold=0.80)

save_cleaned(df, filename, 'RQ3', orig_rows, orig_cols, ops, dropped)


  ✓ Saved: rq3_nasscom_gcc_supply.csv
    8 × 9 → 8 × 9 rows × cols


### RQ3.8 Electronics Production Exports


In [27]:
filename = "rq3_electronics_production_exports.csv"
df = pd.read_csv(DRIVE_RAW / filename, low_memory=False)
orig_rows, orig_cols = len(df), len(df.columns)
ops = []

df = standardize_columns(df)
ops.append('column_standardization')

df, dropped = drop_null_heavy_columns(df, threshold=0.80)

save_cleaned(df, filename, 'RQ3', orig_rows, orig_cols, ops, dropped)


  ✓ Saved: rq3_electronics_production_exports.csv
    21 × 21 → 21 × 21 rows × cols


### RQ3.9 IESA Demand Projections (compiled reference)


In [28]:
filename = "rq3_ism_iesa_demand_projections.csv"
df = pd.read_csv(DRIVE_RAW / filename, low_memory=False)
orig_rows, orig_cols = len(df), len(df.columns)
ops = []

df = standardize_columns(df)
ops.append('column_standardization')

# This is a compiled reference table — minimal cleaning needed
save_cleaned(df, filename, 'RQ3', orig_rows, orig_cols,
             ops + ['compiled_reference_minimal_cleaning'])


  ✓ Saved: rq3_ism_iesa_demand_projections.csv
    3 × 7 → 3 × 7 rows × cols


### RQ3.10 Workforce Compiled (compiled reference)


In [29]:
filename = "rq3_semiconductor_workforce_compiled.csv"
df = pd.read_csv(DRIVE_RAW / filename, low_memory=False)
orig_rows, orig_cols = len(df), len(df.columns)
ops = []

df = standardize_columns(df)
ops.append('column_standardization')

save_cleaned(df, filename, 'RQ3', orig_rows, orig_cols,
             ops + ['compiled_reference_minimal_cleaning'])


  ✓ Saved: rq3_semiconductor_workforce_compiled.csv
    7 × 6 → 7 × 6 rows × cols


---
## RQ4 — Cost Competitiveness (10 sources)

Cleaning targets:
- World Bank Cost: standardize ISO3
- World Bank Electricity: standardize ISO3
- RBI Lending: parse fiscal year strings (in processed/ folder)
- DPIIT FDI: parse year column
- Corporate Tax Rates: standardize country names
- India Power Tariffs: standardize columns
- ISM Project Subsidies: standardize columns
- ISM Scheme Parameters: compiled reference (light cleaning)
- Global Subsidy Comparison: compiled reference (light cleaning)
- KMO/Bartlett readiness check on numeric cost dimensions


### RQ4.1 World Bank Cost


In [30]:
filename = "rq4_worldbank_cost.csv"
df = pd.read_csv(DRIVE_RAW / filename, low_memory=False)
orig_rows, orig_cols = len(df), len(df.columns)
ops = []

df = standardize_columns(df)
ops.append('column_standardization')

# Standardize ISO3 column name
for c in ['economy', 'iso3', 'country']:
    if c in df.columns:
        df = df.rename(columns={c: 'country_iso3'})
        ops.append(f'rename_{c}_to_country_iso3')
        break

df, dropped = drop_null_heavy_columns(df, threshold=0.80,
                                       protect=['country_iso3', 'year'])

save_cleaned(df, filename, 'RQ4', orig_rows, orig_cols, ops, dropped)


  ✓ Saved: rq4_worldbank_cost.csv
    80 × 9 → 80 × 9 rows × cols


### RQ4.2 World Bank Electricity Indicators


In [31]:
filename = "rq4_worldbank_electricity_indicators.csv"
df = pd.read_csv(DRIVE_RAW / filename, low_memory=False)
orig_rows, orig_cols = len(df), len(df.columns)
ops = []

df = standardize_columns(df)
ops.append('column_standardization')

if 'iso3' in df.columns and 'country_iso3' not in df.columns:
    df = df.rename(columns={'iso3': 'country_iso3'})
    ops.append('rename_iso3_to_country_iso3')

df, dropped = drop_null_heavy_columns(df, threshold=0.80,
                                       protect=['country_iso3', 'year', 'indicator', 'value'])

save_cleaned(df, filename, 'RQ4', orig_rows, orig_cols, ops, dropped)


  ✓ Saved: rq4_worldbank_electricity_indicators.csv
    200 × 7 → 200 × 7 rows × cols


### RQ4.3 RBI Lending (from processed/ folder)


In [32]:
filename = "rq4_rbi_lending.csv"
df = pd.read_csv(DRIVE_PROC / filename, low_memory=False)  # Note: processed folder
orig_rows, orig_cols = len(df), len(df.columns)
ops = []

df = standardize_columns(df)
ops.append('column_standardization')

# Parse year (might be fiscal year string like "2024-25")
for col in df.columns:
    if 'year' in col.lower() or 'date' in col.lower():
        if df[col].dtype == 'object':
            # Try to extract numeric year
            df[f'{col}_int'] = pd.to_numeric(
                df[col].astype(str).str.extract(r'(\d{4})')[0], errors='coerce')
            ops.append(f'extract_year_from_{col}')

df, dropped = drop_null_heavy_columns(df, threshold=0.80)

save_cleaned(df, filename, 'RQ4', orig_rows, orig_cols, ops, dropped)


  ✓ Saved: rq4_rbi_lending.csv
    8 × 14 → 8 × 16 rows × cols


### RQ4.4 DPIIT FDI


In [33]:
filename = "rq4_dpiit_fdi.csv"
df = pd.read_csv(DRIVE_RAW / filename, low_memory=False)
orig_rows, orig_cols = len(df), len(df.columns)
ops = []

df = standardize_columns(df)
ops.append('column_standardization')

# Parse year if it's a string
if 'year' in df.columns and df['year'].dtype == 'object':
    df['year'] = pd.to_numeric(df['year'].astype(str).str.extract(r'(\d{4})')[0], errors='coerce')
    ops.append('parse_year_to_int')

df, dropped = drop_null_heavy_columns(df, threshold=0.80, protect=['year'])

save_cleaned(df, filename, 'RQ4', orig_rows, orig_cols, ops, dropped)


  ✓ Saved: rq4_dpiit_fdi.csv
    7 × 5 → 7 × 5 rows × cols


### RQ4.5 Corporate Tax Rates Comparison


In [34]:
filename = "rq4_corporate_tax_rates_comparison.csv"
df = pd.read_csv(DRIVE_RAW / filename, low_memory=False)
orig_rows, orig_cols = len(df), len(df.columns)
ops = []

df = standardize_columns(df)
ops.append('column_standardization')

# Standardize country to ISO3
for c in ['country', 'iso3']:
    if c in df.columns:
        if c == 'country':
            df['country_iso3'] = df['country'].apply(standardize_iso3)
            ops.append('iso3_from_country')
        elif c == 'iso3' and 'country_iso3' not in df.columns:
            df = df.rename(columns={'iso3': 'country_iso3'})
        break

df, dropped = drop_null_heavy_columns(df, threshold=0.80,
                                       protect=['country_iso3', 'country'])

save_cleaned(df, filename, 'RQ4', orig_rows, orig_cols, ops, dropped)


  ✓ Saved: rq4_corporate_tax_rates_comparison.csv
    10 × 9 → 10 × 10 rows × cols


### RQ4.6 India Power Tariffs


In [35]:
filename = "rq4_india_power_tariffs.csv"
df = pd.read_csv(DRIVE_RAW / filename, low_memory=False)
orig_rows, orig_cols = len(df), len(df.columns)
ops = []

df = standardize_columns(df)
ops.append('column_standardization')

df, dropped = drop_null_heavy_columns(df, threshold=0.80)

save_cleaned(df, filename, 'RQ4', orig_rows, orig_cols, ops, dropped)


  ✓ Saved: rq4_india_power_tariffs.csv
    93 × 14 → 93 × 12 rows × cols


### RQ4.7 ISM Project Subsidies


In [36]:
filename = "rq4_ism_project_subsidies.csv"
df = pd.read_csv(DRIVE_RAW / filename, low_memory=False)
orig_rows, orig_cols = len(df), len(df.columns)
ops = []

df = standardize_columns(df)
ops.append('column_standardization')

if 'state' in df.columns:
    df['state'] = df['state'].apply(standardize_state)
    ops.append('state_name_standardization')

save_cleaned(df, filename, 'RQ4', orig_rows, orig_cols, ops)


  ✓ Saved: rq4_ism_project_subsidies.csv
    6 × 7 → 6 × 7 rows × cols


### RQ4.8 ISM Scheme Parameters (compiled reference)


In [37]:
filename = "rq4_ism_scheme_parameters.csv"
df = pd.read_csv(DRIVE_RAW / filename, low_memory=False)
orig_rows, orig_cols = len(df), len(df.columns)
ops = []

df = standardize_columns(df)
ops.append('column_standardization')

save_cleaned(df, filename, 'RQ4', orig_rows, orig_cols,
             ops + ['compiled_reference_minimal_cleaning'])


  ✓ Saved: rq4_ism_scheme_parameters.csv
    6 × 8 → 6 × 8 rows × cols


### RQ4.9 Global Subsidy Comparison (compiled reference)


In [38]:
filename = "rq4_global_subsidy_comparison.csv"
df = pd.read_csv(DRIVE_RAW / filename, low_memory=False)
orig_rows, orig_cols = len(df), len(df.columns)
ops = []

df = standardize_columns(df)
ops.append('column_standardization')

save_cleaned(df, filename, 'RQ4', orig_rows, orig_cols,
             ops + ['compiled_reference_minimal_cleaning'])


  ✓ Saved: rq4_global_subsidy_comparison.csv
    6 × 8 → 6 × 8 rows × cols


### RQ4.10 Source 07 FDI Sectorwise (raw API output)


In [39]:
filename = "source07_fdi_sectorwise_api.csv"
df = pd.read_csv(DRIVE_RAW / filename, low_memory=False)
orig_rows, orig_cols = len(df), len(df.columns)
ops = []

df = standardize_columns(df)
ops.append('column_standardization')

df, dropped = drop_null_heavy_columns(df, threshold=0.80)

save_cleaned(df, filename, 'RQ4', orig_rows, orig_cols, ops, dropped)


  ✓ Saved: source07_fdi_sectorwise_api.csv
    122 × 13 → 122 × 13 rows × cols


---
## Diagnostic Flags

These tests **flag potential issues** for awareness — they don't transform the data.  
Transformations (first-differencing for RQ1, normalization for RQ4) happen in Script 5 (Modeling).


### Diagnostic 1: Stationarity check for RQ1 monthly series


In [40]:
# Run ADF test on monthly DGFT import series to check stationarity
from statsmodels.tsa.stattools import adfuller, kpss

dgft = pd.read_csv(DRIVE_CLEAN / "rq1_dgft_tradestat.csv", low_memory=False)

print("RQ1 STATIONARITY CHECK (Augmented Dickey-Fuller)")
print("=" * 70)
print("H0: series has unit root (non-stationary)")
print("H1: series is stationary")
print("Reject H0 if p < 0.05 → series is stationary")
print()

# Check on aggregated monthly imports if column exists
if 'import_value_usd_million' in dgft.columns and 'year_month' in dgft.columns:
    monthly_imports = dgft.groupby('year_month')['import_value_usd_million'].sum().sort_index()

    if len(monthly_imports) >= 24:
        try:
            adf_result = adfuller(monthly_imports.dropna(), autolag='AIC')
            kpss_result = kpss(monthly_imports.dropna(), regression='c', nlags='auto')

            print(f"Monthly imports time series ({len(monthly_imports)} months)")
            print(f"  ADF p-value:  {adf_result[1]:.4f} {'(stationary)' if adf_result[1] < 0.05 else '(NON-stationary — flag for differencing)'}")
            print(f"  KPSS p-value: {kpss_result[1]:.4f} {'(non-stationary)' if kpss_result[1] < 0.05 else '(stationary)'}")

            stat_flag = "stationarity_OK" if adf_result[1] < 0.05 else "non_stationary_consider_first_differencing"
            cleaning_report.append({
                'filename': 'rq1_dgft_tradestat.csv',
                'rq': 'RQ1',
                'original_rows': len(monthly_imports),
                'original_cols': 1,
                'cleaned_rows': len(monthly_imports),
                'cleaned_cols': 1,
                'rows_changed': 0, 'cols_changed': 0,
                'operations_applied': 'stationarity_diagnostic',
                'columns_dropped': '', 'rows_filtered_reason': '',
                'flags': f'ADF p={adf_result[1]:.4f}; KPSS p={kpss_result[1]:.4f}; {stat_flag}',
                'cleaning_date': datetime.now().strftime('%Y-%m-%d %H:%M:%S')
            })
        except Exception as e:
            print(f"  ⚠ Stationarity test failed: {e}")
    else:
        print(f"  ⚠ Insufficient observations ({len(monthly_imports)}) for stationarity test")
else:
    print("  ⚠ Required columns not found in cleaned DGFT data")


RQ1 STATIONARITY CHECK (Augmented Dickey-Fuller)
H0: series has unit root (non-stationary)
H1: series is stationary
Reject H0 if p < 0.05 → series is stationary

  ⚠ Required columns not found in cleaned DGFT data


### Diagnostic 2: KMO/Bartlett readiness check for RQ4 PCA


In [ ]:
# Check KMO and Bartlett's test on RQ4 cost dimensions
from factor_analyzer.factor_analyzer import calculate_kmo, calculate_bartlett_sphericity

print("RQ4 PCA READINESS CHECK")
print("=" * 70)
print("KMO target: > 0.6 (sampling adequacy for PCA)")
print("Bartlett's target: p < 0.05 (correlations exist between variables)")
print()

# Build a simple cost matrix for the diagnostic
# This is just a readiness check — actual PCA happens in Script 5
try:
    wb_cost = pd.read_csv(DRIVE_CLEAN / "rq4_worldbank_cost.csv", low_memory=False)

    # Pick numeric columns that look like cost dimensions
    num_cols = wb_cost.select_dtypes(include=[np.number]).columns.tolist()
    cost_cols = [c for c in num_cols if c not in ['year']][:6]  # take up to 6 numeric cols

    if len(cost_cols) >= 3:
        cost_matrix = wb_cost[cost_cols].dropna()

        if len(cost_matrix) >= 10 and len(cost_cols) >= 3:
            kmo_all, kmo_model = calculate_kmo(cost_matrix)
            chi_sq, p_value = calculate_bartlett_sphericity(cost_matrix)

            print(f"Cost matrix: {len(cost_matrix)} observations × {len(cost_cols)} variables")
            print(f"Variables tested: {cost_cols}")
            print()
            print(f"  KMO (overall):    {kmo_model:.3f} {'✓ ADEQUATE' if kmo_model > 0.6 else '⚠ INADEQUATE — fall back to equal-weighted CCI'}")
            print(f"  Bartlett's p:     {p_value:.6f} {'✓ correlations exist' if p_value < 0.05 else '⚠ no correlations — PCA may not be useful'}")

            kmo_flag = "PCA_ready" if kmo_model > 0.6 else "PCA_marginal_consider_equal_weights"

            cleaning_report.append({
                'filename': 'rq4_worldbank_cost.csv',
                'rq': 'RQ4',
                'original_rows': len(cost_matrix),
                'original_cols': len(cost_cols),
                'cleaned_rows': len(cost_matrix),
                'cleaned_cols': len(cost_cols),
                'rows_changed': 0, 'cols_changed': 0,
                'operations_applied': 'PCA_readiness_diagnostic',
                'columns_dropped': '', 'rows_filtered_reason': '',
                'flags': f'KMO={kmo_model:.3f}; Bartlett p={p_value:.6f}; {kmo_flag}',
                'cleaning_date': datetime.now().strftime('%Y-%m-%d %H:%M:%S')
            })
        else:
            print(f"  ⚠ Insufficient data: {len(cost_matrix)} obs × {len(cost_cols)} cols")
    else:
        print(f"  ⚠ Need at least 3 numeric cost columns; found {len(cost_cols)}")
except Exception as e:
    print(f"  ⚠ KMO/Bartlett test failed: {e}")
    print("  This may resolve once Script 4 (merge) creates the full RQ4 panel")


---
## Cleaning Report Export


In [ ]:
# Save the cleaning report as CSV
report_df = pd.DataFrame(cleaning_report)
report_path = DRIVE_CLEAN / "cleaning_report.csv"
report_df.to_csv(report_path, index=False)

print("=" * 70)
print(f"CLEANING REPORT SAVED: {report_path}")
print("=" * 70)
print(f"Total cleaning operations recorded: {len(report_df)}")
print()
print("Files cleaned per RQ:")
print(report_df.groupby('rq').size().to_string())
print()
print("Files with flags (warnings):")
flagged = report_df[report_df['flags'].astype(bool) & (report_df['flags'] != '')]
if len(flagged) > 0:
    for _, row in flagged.iterrows():
        print(f"  ⚠ {row['filename']}: {row['flags']}")
else:
    print("  (none)")

print()
print("Total columns dropped across all files:")
total_dropped = sum(len(r.split('; ')) for r in report_df['columns_dropped'] if r)
print(f"  {total_dropped} null-heavy columns dropped")

print()
print("Display first 5 rows of cleaning report:")
print(report_df[['filename', 'rq', 'original_rows', 'cleaned_rows', 'original_cols', 'cleaned_cols']].head().to_string(index=False))


---
## Final Verification


In [ ]:
# Verify all 32 cleaned files exist
print("=" * 70)
print("CLEANING COMPLETE — FINAL VERIFICATION")
print("=" * 70)

cleaned_files = sorted(DRIVE_CLEAN.glob("*.csv"))
expected_count = 32

print(f"\nFiles in cleaned/ folder: {len(cleaned_files)}")
print(f"Expected count: {expected_count} (32 datasets) + 1 cleaning_report.csv = 33")
print()

# Compare with raw + processed source counts
raw_count = len(list(DRIVE_RAW.glob("*.csv")))
proc_count = len(list(DRIVE_PROC.glob("*.csv"))) if DRIVE_PROC.exists() else 0
source_total = raw_count + proc_count

print(f"Source CSVs: {source_total}")
print(f"Cleaned CSVs: {len(cleaned_files) - 1}  (excluding cleaning_report.csv)")
print(f"Match: {'✓ YES' if (len(cleaned_files) - 1) == source_total else '✗ NO — investigate gaps'}")
print()

# List cleaned files
print("Cleaned files by RQ:")
rq_groups = {'RQ1': [], 'RQ2': [], 'RQ3': [], 'RQ4': [], 'Other': []}
for f in cleaned_files:
    n = f.stem.lower()
    if n == 'cleaning_report':
        continue
    if n.startswith('rq1') or 'rq1' in n:
        rq_groups['RQ1'].append(f.name)
    elif n.startswith('rq2') or 'rq2' in n:
        rq_groups['RQ2'].append(f.name)
    elif n.startswith('rq3') or 'rq3' in n:
        rq_groups['RQ3'].append(f.name)
    elif n.startswith('rq4') or 'rq4' in n:
        rq_groups['RQ4'].append(f.name)
    else:
        rq_groups['Other'].append(f.name)

for rq, files in rq_groups.items():
    if files:
        print(f"\n{rq} ({len(files)} files):")
        for f in files:
            print(f"  ✓ {f}")

print()
print("=" * 70)
print("NEXT: Run Script 4 to merge cleaned files into 1 panel per RQ")
print("=" * 70)
